# To dos:
- create subagents + permissions
- add backend
- add skill + middleware 
- tools: git commit
- SANDBOX + HITL

# Test tools

In [2]:
# test all the ingestion tools 
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # when notebook is in ./notebooks
sys.path.insert(0, str(repo_root))

#from src.tools.ingest_tools import fetch_arxiv, parse_pdf, fetch_web_article, lint_check



## fetch_arxiv

In [2]:
doc = fetch_arxiv("attention is all you need")
doc

{'title': 'Attention Is All You Need',
 'authors': ['Ashish Vaswani',
  'Noam Shazeer',
  'Niki Parmar',
  'Jakob Uszkoreit',
  'Llion Jones',
  'Aidan N. Gomez',
  'Lukasz Kaiser',
  'Illia Polosukhin'],
 'abstract': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our mod

## Docling

In [ ]:
from src.tools.parsers.docling_parser import parse_pdf_docling
parsed = parse_pdf_docling("/Users/dangphuonganh/Documents/llm_wiki/raw/papers/attention_is_all_you_need.pdf", parse_images=True, save_table_images=True)

print(parsed["title"])
print(parsed["page_count"])
print(f"{parsed['table_blocks']} tables found")
print(f"{parsed['images']} figures extracted")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Attention Is All You Need
15
4 tables found
6 figures extracted


In [3]:
print("body_chars:", parsed.get("body_chars"))
print("markdown_path:", parsed.get("markdown_path"))

## Attention Is All You Need

Ashish Vaswani ∗ Google Brain avaswani@google.com

Noam Shazeer ∗ Google Brain noam@google.com

Llion Jones ∗ Google Research llion@google.com

Niki Parmar ∗ Google Research nikip@google.com

Aidan N. Gomez ∗ † University of Toronto aidan@cs.toronto.edu

Jakob Uszkoreit ∗ Google Research usz@google.com

Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com

∗ ‡

Illia Polosukhin illia.polosukhin@gmail.com

## 

## 1 Introduction

Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].

Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning th

## Pymupdf -- not as good as docling

In [9]:
doc = parse_pdf("/Users/dangphuonganh/Documents/llm_wiki/raw/papers/attention_is_all_you_need.pdf") #parse_images=True)
doc

{'title': 'Provided proper attribution is provided, Google hereby grants permission to',
 'abstract': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs,

## Check lint tool

In [1]:
# test all the ingestion tools 
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # when notebook is in ./notebooks
sys.path.insert(0, str(repo_root))

from src.tools.lint import lint_check
lint_check()

'lint: OK'

# Agents

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
repo_root = _cwd if (_cwd / "skills").is_dir() else _cwd.parent
sys.path.insert(0, str(repo_root))

# Real filesystem directories (no /** glob)
wiki_dir     = repo_root / "wiki"
skills_dir   = repo_root / "skills"
raw_dir      = repo_root / "raw"
memories_dir = repo_root / "memories"

print(f"repo_root: {repo_root}")
print(f"wiki_dir:  {wiki_dir}")
print(f"skills_dir: {skills_dir}")

from deepagents import create_deep_agent, FilesystemPermission
from deepagents.backends import CompositeBackend, StateBackend, FilesystemBackend
from langchain.chat_models import init_chat_model
from src.tools.ingest_tools import parse_pdf_docling, fetch_arxiv, lint_check
from src.prompts.system_prompt import INGEST_AGENT_SYSTEM_PROMPT, PHASE_1_SUPERVISOR_PROMPT

ingest_subagent = {
    "name": "ingest",
    "skills": ["/skills/paper-ingestion/"],
    "permissions": [
        FilesystemPermission(paths=["/skills/**"],    operations=["read"],         mode="allow"),
        FilesystemPermission(paths=["/raw/assests/", "/raw/assets/**"],       operations=["read", "write"], mode="allow"),
        FilesystemPermission(paths=["/wiki/","/wiki/**"],      operations=["read", "write"], mode="allow"),
        FilesystemPermission(paths=["/memories/**"],  operations=["read"],         mode="allow"),
        FilesystemPermission(paths=["/workspace/**"], operations=["read", "write"], mode="allow"),
        FilesystemPermission(paths=["/**"],           operations=["read", "write"], mode="deny"),
    ],
    "description": "Paper ingestion, wiki maintenance, citation extract",
    "system_prompt": INGEST_AGENT_SYSTEM_PROMPT,
    "tools": [fetch_arxiv, parse_pdf_docling, lint_check],
}

model = "claude-haiku-4-5-20251001"
_llm = init_chat_model(model, max_retries=8, timeout=120.0)

agent = create_deep_agent(
    model=_llm,
    skills=["/skills/"],
    memory=["/memories/AGENTS.md"],
    system_prompt=PHASE_1_SUPERVISOR_PROMPT,
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/raw/":      FilesystemBackend(root_dir=str(raw_dir),      virtual_mode=True),
            "/wiki/":     FilesystemBackend(root_dir=str(wiki_dir),     virtual_mode=True),
            "/skills/":   FilesystemBackend(root_dir=str(skills_dir),   virtual_mode=True),
            "/memories/": FilesystemBackend(root_dir=str(memories_dir), virtual_mode=True),
        },
    ),
    subagents=[ingest_subagent],
    tools=[lint_check],
)

repo_root: /Users/dangphuonganh/Documents/llm_wiki
wiki_dir:  /Users/dangphuonganh/Documents/llm_wiki/wiki
skills_dir: /Users/dangphuonganh/Documents/llm_wiki/skills


In [2]:
#query = "create wiki for the Attention is all you need paper we already fetched and parsed in raw/"
query = "Run lint check on the entire wiki"
# query = "tell ingest subagent to check and fix any lint errors in the wiki"

In [ ]:
# Test it
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "Ingest the paper at https://arxiv.org/abs/1706.03762"}]
# })

In [17]:
# Paths are relative to the kernel CWD (often notebooks/), not this file.
# skills/ lives at the repo root — resolve explicitly:
from pathlib import Path
import os

# _cwd = Path.cwd()
# repo_root = _cwd if (_cwd / "skills").is_dir() else _cwd.parent
skills = repo_root / "skills"
print("repo_root:", repo_root.resolve())
print(os.listdir(skills))  # skill folders
print(os.listdir(skills / "paper-ingestion"))  # e.g. SKILL.md

repo_root: /Users/dangphuonganh/Documents/llm_wiki
['.DS_Store', 'paper-ingestion']
['SKILL.md']


In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
repo_root = _cwd if (_cwd / "skills").is_dir() else _cwd.parent
sys.path.insert(0, str(repo_root))
print(repo_root)

from src.agents.agent import agent


/Users/dangphuonganh/Documents/llm_wiki
repo_root: /Users/dangphuonganh/Documents/llm_wiki
wiki_dir:  /Users/dangphuonganh/Documents/llm_wiki/wiki
skills_dir: /Users/dangphuonganh/Documents/llm_wiki/skills


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
repo_root = _cwd if (_cwd / "skills").is_dir() else _cwd.parent
sys.path.insert(0, str(repo_root))

from src.agents.agent import agent

current_source = ""
query = "https://arxiv.org/pdf/1608.05859"

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode=["updates", "messages"],
    subgraphs=True,
    version="v2",
):  
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        # Check if this event came from a subagent (namespace contains "tools:")
        is_subagent = any(s.startswith("tools:") for s in chunk["ns"])

        if is_subagent:
            # Token from a subagent
            subagent_ns = next(s for s in chunk["ns"] if s.startswith("tools:"))
            if subagent_ns != current_source:
                print(f"\n\n--- [subagent: {subagent_ns}] ---")
                current_source = subagent_ns
            if token.content:
                print(token.content, end="", flush=True)
        else:
            # Token from the main agent
            if "main" != current_source:
                print("\n\n--- [main agent] ---")
                current_source = "main"
            if token.content:
                print(token.content, end="", flush=True)
                
    if chunk["type"] == "updates":
        # Main agent updates (empty namespace)
        if not chunk["ns"]:
            for node_name, data in chunk["data"].items():
                if node_name == "tools":
                    # Subagent results returned to main agent
                    for msg in data.get("messages", []):
                        if msg.type == "tool":
                            print(f"\nSubagent complete: {msg.name}")
                            print(f"  Result: {str(msg.content)[:200]}...")
                else:
                    print(f"[main agent] step: {node_name}")

        # Subagent updates (non-empty namespace)
        else:
            for node_name, data in chunk["data"].items():
                print(f"  [{chunk['ns'][0]}] step: {node_name}")

repo_root: /Users/dangphuonganh/Documents/llm_wiki
wiki_dir:  /Users/dangphuonganh/Documents/llm_wiki/wiki
skills_dir: /Users/dangphuonganh/Documents/llm_wiki/skills
[main agent] step: SkillsMiddleware.before_agent
[main agent] step: PatchToolCallsMiddleware.before_agent
[main agent] step: MemoryMiddleware.before_agent


--- [main agent] ---
[{'text': "I'll delegate", 'type': 'text', 'index': 0}][{'text': ' this paper ingestion to the ingest subagent.', 'type': 'text', 'index': 0}][{'id': 'toolu_012dM2QjxpGrAKL1aE2K3k9o', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'task', 'type': 'tool_use', 'index': 1}][{'partial_json': '', 'type': 'input_json_delta', 'index': 1}][{'partial_json': '{"descri', 'type': 'input_json_delta', 'index': 1}][{'partial_json': 'ption": "I', 'type': 'input_json_delta', 'index': 1}][{'partial_json': 'ngest the', 'type': 'input_json_delta', 'index': 1}][{'partial_json': ' pape', 'type': 'input_json_delta', 'index': 1}][{'partial_json': 'r fr', 'type': 'inpu

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)